# Chapter 11: Data Cleaning and Basic Analysis

**B.Pharm companion Jupyter notebook — worked examples and concept demonstrations**

## Chapter focus

Inspect raw data, document conversions and missing-value rules, remove confirmed duplicate identifiers, select, filter, sort, summarize and export.

## Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

## Important notes and how to use this notebook

1. Use a Python 3 kernel. Read the problem and algorithm, predict the result, then run the cell with **Shift+Enter**.
2. Choose **Restart Kernel and Run All Cells** to reproduce the saved outputs. Run from top to bottom; later demonstrations may use earlier variables.
3. Outputs below code cells were produced by execution. Paths, versions, memory diagnostics and set display order can differ between computers.
4. All records are simulated for education. Retain units and distinguish a descriptive result from a clinical or regulatory decision.
5. Files written by examples are kept in `outputs/chapter_11`. Rerunning a file-writing example replaces its own generated output.

**Packages:** `pandas`, `openpyxl`. If required, run this once in a separate cell and restart the kernel:

```python
%pip install pandas openpyxl
```

Use pandas 2.0 or later for mixed-format date parsing.

**Data files:** `adr_reports_raw.csv`, `pharma_quality_clean_expected.csv`, `pharma_quality_raw.csv`.

Extract the Chapter 11 dataset ZIP beside this notebook, preserving `data/chapter_11`. Individual data files can also be placed in the same folder as the notebook. The setup cell searches both locations.

This chapter uses `pharma_quality_raw.csv` as its main worked dataset. `adr_reports_raw.csv` is a separate practice dataset; `pharma_quality_clean_expected.csv` is a reference result, not an input to the cleaning workflow.


In [1]:
from pathlib import Path

WORKSHOP_ROOT = Path.cwd().resolve()
OUTPUT_DIR = WORKSHOP_ROOT / "outputs" / "chapter_11"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_files = ['adr_reports_raw.csv', 'pharma_quality_clean_expected.csv', 'pharma_quality_raw.csv']
candidates = [WORKSHOP_ROOT / "data" / "chapter_11", WORKSHOP_ROOT]
DATA_DIR = next((folder for folder in candidates
                 if all((folder / name).is_file() for name in required_files)), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Extract the chapter dataset ZIP beside this notebook or put all listed data files in the notebook folder."
    )
print("Companion data files found:", len(required_files))

print("Chapter 11 ready. Generated files: outputs/chapter_11")

Companion data files found: 3
Chapter 11 ready. Generated files: outputs/chapter_11


## Contents

- [01. 11.4.2 Installing or Verifying pandas](#example-11-1)
- [02. 11.4.3 Downloading and Placing the Companion Data](#example-11-2)
- [03. 11.6 Importing and Inspecting a Dataset Before Cleaning](#example-11-3)
- [04. Program 1: Initial Data Quality Inspection](#example-11-4)
- [05. 11.7.1 Detecting Missing Values](#example-11-5)
- [06. 11.7.2 Dropping Missing Values with dropna()](#example-11-6)
- [07. 11.7.3 Filling Missing Values with fillna()](#example-11-7)
- [08. Program 2: Handling Missing Numerical Values with a Documented Median Rule](#example-11-8)
- [09. 11.8 Duplicate Records](#example-11-9)
- [10. Program 3: Removing a Repeated Record Identifier](#example-11-10)
- [11. 11.9 Incorrect Data Types and Type Conversion](#example-11-11)
- [12. Program 4: Correcting Text, Numeric, and Date Fields](#example-11-12)
- [13. 11.10 Selecting Rows and Columns](#example-11-13)
- [14. Program 5: Selecting a Compact Quality Review Table](#example-11-14)
- [15. 11.11 Conditional Filtering](#example-11-15)
- [16. Program 6: Filtering Records for a Classroom Attention Screen](#example-11-16)
- [17. 11.12 Sorting Data](#example-11-17)
- [18. Program 7: Sorting by Site and Assay](#example-11-18)
- [19. 11.13 Grouping and Aggregation](#example-11-19)
- [20. Program 8: Product-wise Aggregation of Assay Values](#example-11-20)
- [21. 11.13.1 Multiple Grouping Columns](#example-11-21)
- [22. 11.14 Exporting Cleaned Datasets](#example-11-22)
- [23. Program 9: Exporting a Cleaned Pharmaceutical Dataset](#example-11-23)
- [24. Program 10: Complete Cleaning, Validation, Summary, and Export](#example-11-24)
- [25. 11.16 Validation After Cleaning](#example-11-25)

<a id="example-11-1"></a>

## 01. 11.4.2 Installing or Verifying pandas

### Problem statement

Demonstrate installing or Verifying pandas using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support installing or verifying pandas and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Import `importlib.metadata`.
2. Display the results for inspection.
3. Import `pandas`.

### Important notes

Installation is optional preparation, so it is shown in the opening notes. This executable cell verifies installed versions without changing the environment.

### Code and executed output

The output appears directly below the code cell.

In [2]:
from importlib.metadata import version
print("pandas:", version("pandas"))
print("openpyxl:", version("openpyxl"))
import pandas as pd

pandas: 2.2.3
openpyxl: 3.1.5


### Explanation

The %pip command installs packages into the Jupyter environment. import pandas as pd loads pandas using the standard alias pd. The version number may differ between computers; successful import without an error is the important check.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-2"></a>

## 02. 11.4.3 Downloading and Placing the Companion Data

### Problem statement

Demonstrate downloading and Placing the Companion Data using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support downloading and placing the companion data and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Import `pathlib`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [3]:
from pathlib import Path
print(Path.cwd())

/workspace/scratch/1ff152582af4/notebook_work/execution/chapter_11


### Explanation

Path.cwd() shows the notebook working directory. A relative file path is interpreted from this location, so checking the path is one of the first steps when FileNotFoundError occurs.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-3"></a>

## 03. 11.6 Importing and Inspecting a Dataset Before Cleaning

### Problem statement

Demonstrate importing and Inspecting a Dataset Before Cleaning using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support importing and inspecting a dataset before cleaning and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `pd.read_csv(DATA_DIR / 'pharma_quality_raw.csv')` in `df`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [4]:
df = pd.read_csv(DATA_DIR / "pharma_quality_raw.csv")
print(df.shape)
print(df[["assay_percent", "dissolution_percent"]].dtypes)
print(df[["assay_percent", "dissolution_percent"]].isna().sum())

(20, 13)
assay_percent          object
dissolution_percent    object
dtype: object
assay_percent          1
dissolution_percent    1
dtype: int64


### Explanation

The dataset has 20 rows and 13 columns. assay_percent and dissolution_percent are read as object because text values are mixed with numbers. Missing-value counts show where cleaning is required. In pandas, a blank CSV field is commonly imported as NaN.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-4"></a>

## 04. Program 1: Initial Data Quality Inspection

### Problem statement

Import the raw pharmaceutical-quality CSV file and report its dimensions, column names, selected data types, missing-value counts, and duplicate count.

### Objective

To identify the main data-quality issues that must be addressed before analysis.

### Pharmacy context

The output shows that cleaning is required in assay, dissolution, humidity, and review status. It also shows one exact duplicate. These observations justify later cleaning steps; they are not themselves evidence that a batch passes or fails any real quality specification.

### Algorithm

1. Import pandas.
2. Read the raw CSV file into a DataFrame named quality_df.
3. Display the first five records.
4. Report rows and columns using shape.
5. Report data types using dtypes.
6. Count missing values using isna().sum().
7. Count exact duplicate rows using duplicated().sum().

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| quality_df | Raw pharmaceutical quality DataFrame | 20 rows x 13 columns |
| shape | Tuple containing row and column count | (20, 13) |
| isna() | Marks missing values | True / False |
| duplicated() | Marks repeated full rows | True / False |

### Code and executed output

The output appears directly below the code cell.

In [5]:
import pandas as pd

quality_df = pd.read_csv(DATA_DIR / "pharma_quality_raw.csv")

print("Shape:", quality_df.shape)
print("Missing values:")
print(quality_df.isna().sum()[quality_df.isna().sum() > 0])
print("Exact duplicate rows:", quality_df.duplicated().sum())

Shape: (20, 13)
Missing values:
assay_percent          1
dissolution_percent    1
humidity_percent       1
review_status          1
dtype: int64
Exact duplicate rows: 1


### Explanation

shape returns (rows, columns). isna() identifies missing cells and sum() counts them column-wise. The expression inside brackets keeps only columns whose missing-value count is greater than zero. duplicated().sum() reports repeated complete rows.

### Interpretation

The output shows that cleaning is required in assay, dissolution, humidity, and review status. It also shows one exact duplicate. These observations justify later cleaning steps; they are not themselves evidence that a batch passes or fails any real quality specification.

<a id="example-11-5"></a>

## 05. 11.7.1 Detecting Missing Values

### Problem statement

Demonstrate detecting Missing Values using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support detecting missing values and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Execute `quality_df['assay_percent'].isna().sum()`.

### Code and executed output

The output appears directly below the code cell.

In [6]:
quality_df["assay_percent"].isna().sum()

np.int64(1)

### Explanation

isna() returns True where a value is missing. Calling sum() on the Boolean result counts True values. Here one assay entry is blank in the raw CSV.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-6"></a>

## 06. 11.7.2 Dropping Missing Values with dropna()

### Problem statement

Demonstrate dropping Missing Values with dropna() using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support dropping missing values with dropna() and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `quality_df.dropna(subset=['assay_percent'])` in `complete_assay`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [7]:
complete_assay = quality_df.dropna(subset=["assay_percent"])
print(len(quality_df), len(complete_assay))

20 19


### Explanation

Only rows missing assay_percent are removed. The original DataFrame remains unchanged because the result is assigned to complete_assay. subset prevents unrelated missing fields from causing row deletion.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-7"></a>

## 07. 11.7.3 Filling Missing Values with fillna()

### Problem statement

Demonstrate filling Missing Values with fillna() using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support filling missing values with fillna() and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `pd.to_numeric(quality_df['assay_percent'].astype('string').str.replace('%', '', regex=False), errors='coerce')` in `assay_numeric`.
2. Store `assay_numeric.fillna(assay_numeric.median())` in `assay_filled`.
3. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [8]:
assay_numeric = pd.to_numeric(quality_df["assay_percent"].astype("string").str.replace("%", "", regex=False), errors="coerce")
assay_filled = assay_numeric.fillna(assay_numeric.median())
print(round(assay_numeric.median(), 2))

99.6


### Explanation

to_numeric first converts usable text to numbers and turns non-convertible entries into missing values because errors="coerce". median() then calculates the middle observed value, and fillna() replaces the missing assay value with that median in the demonstration series.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-8"></a>

## 08. Program 2: Handling Missing Numerical Values with a Documented Median Rule

### Problem statement

Clean assay, dissolution, and humidity fields so that simple numerical summaries can be performed without losing entire rows.

### Objective

To demonstrate controlled conversion and median-based replacement while preserving a reproducible record of the chosen replacement values.

### Pharmacy context

In this standalone demonstration, the median assay is approximately 99.60%, the median dissolution value is 86.50%, and the median humidity is 52.00%. These values are calculated only from the simulated records present at this stage. Because errors="coerce" can create new missing values from invalid text, the converted columns must always be checked before and after filling.

### Algorithm

1. Copy the raw DataFrame.
2. Remove the percent sign from assay text.
3. Convert assay, dissolution, and humidity to numeric values using errors="coerce".
4. Calculate the median of each column while ignoring missing values.
5. Fill missing values with the corresponding median.
6. Print the replacement values and the remaining missing counts.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| assay_percent | Assay after numeric conversion | % |
| dissolution_percent | Dissolution after numeric conversion | % |
| humidity_percent | Relative humidity | % |
| errors="coerce" | Invalid numeric text becomes NaN | Conversion parameter |
| median() | Middle observed value | Classroom replacement statistic |

### Important notes

The replacement medians are printed as the source algorithm requests. They are computed at this demonstration stage, before the duplicate-removal workflow.

### Code and executed output

The output appears directly below the code cell.

In [9]:
missing_demo_df = quality_df.copy()

missing_demo_df["assay_percent"] = pd.to_numeric(
    missing_demo_df["assay_percent"]
    .astype("string")
    .str.replace("%", "", regex=False),
    errors="coerce"
)
missing_demo_df["dissolution_percent"] = pd.to_numeric(
    missing_demo_df["dissolution_percent"],
    errors="coerce"
)
missing_demo_df["humidity_percent"] = pd.to_numeric(
    missing_demo_df["humidity_percent"],
    errors="coerce"
)

replacement_values = {}
for col in [
    "assay_percent",
    "dissolution_percent",
    "humidity_percent"
]:
    replacement_values[col] = missing_demo_df[col].median()
    missing_demo_df[col] = missing_demo_df[col].fillna(replacement_values[col])

print(
    missing_demo_df[[
        "assay_percent",
        "dissolution_percent",
        "humidity_percent"
    ]].isna().sum()
)
print("Median replacement values:", {key: float(value) for key, value in replacement_values.items()})

assay_percent          0
dissolution_percent    0
humidity_percent       0
dtype: int64
Median replacement values: {'assay_percent': 99.6, 'dissolution_percent': 86.5, 'humidity_percent': 52.0}


### Explanation

A separate copy named missing_demo_df is used so that the imported raw dataset remains unchanged. str.replace() removes the literal % symbol from assay text. errors="coerce" converts non-numeric entries such as "not recorded" to NaN instead of stopping the program. fillna(column.median()) then applies the documented classroom replacement rule to the demonstration copy.

### Interpretation

In this standalone demonstration, the median assay is approximately 99.60%, the median dissolution value is 86.50%, and the median humidity is 52.00%. These values are calculated only from the simulated records present at this stage. Because errors="coerce" can create new missing values from invalid text, the converted columns must always be checked before and after filling.

<a id="example-11-9"></a>

## 09. 11.8 Duplicate Records

### Problem statement

Demonstrate duplicate Records using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support duplicate records and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [10]:
print(quality_df.duplicated().sum())
print(quality_df.duplicated(subset=["record_id"]).sum())

1
1


### Explanation

The first count checks complete rows. The second checks repeated record_id values. subset identifies the columns that define duplication. keep="first" retains the first occurrence and removes later occurrences.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-10"></a>

## 10. Program 3: Removing a Repeated Record Identifier

### Problem statement

Remove later occurrences of the same record_id while retaining the first occurrence.

### Objective

To prevent an accidentally repeated classroom record from being counted twice in later summaries.

### Pharmacy context

The row count decreases by one, matching the earlier duplicate audit. In a real laboratory or pharmacovigilance dataset, repeated identifiers must be investigated before removal because repeat measurements, follow-up reports, or amended records may be valid rather than duplicates.

### Algorithm

1. Create a working copy of the raw DataFrame named clean_df.
2. Count the rows before duplicate removal.
3. Use drop_duplicates() with subset=["record_id"] and keep="first".
4. Reset the row index after removal.
5. Compare the number of rows before and after cleaning.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| subset | Columns used to decide whether rows are duplicates | ["record_id"] |
| keep | Which occurrence is retained | "first" |
| reset_index(drop=True) | Creates a new sequential index | 0, 1, 2, ... |

### Code and executed output

The output appears directly below the code cell.

In [11]:
clean_df = quality_df.copy()

before = len(clean_df)
clean_df = clean_df.drop_duplicates(
    subset=["record_id"],
    keep="first"
).reset_index(drop=True)
after = len(clean_df)

print("Rows before:", before)
print("Rows after:", after)

Rows before: 20
Rows after: 19


### Explanation

drop_duplicates() searches only the column named in subset rather than comparing every column. keep="first" retains the first R005 record and removes the later repeated occurrence. reset_index(drop=True) creates a new sequential index without adding the old index as a separate column. The resulting clean_df is used as the working DataFrame in the following examples.

### Interpretation

The row count decreases by one, matching the earlier duplicate audit. In a real laboratory or pharmacovigilance dataset, repeated identifiers must be investigated before removal because repeat measurements, follow-up reports, or amended records may be valid rather than duplicates.

<a id="example-11-11"></a>

## 11. 11.9 Incorrect Data Types and Type Conversion

### Problem statement

Demonstrate incorrect Data Types and Type Conversion using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support incorrect data types and type conversion and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `pd.to_numeric(clean_df['assay_percent'], errors='coerce')` in `numeric_assay`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [12]:
numeric_assay = pd.to_numeric(clean_df["assay_percent"], errors="coerce")
print(numeric_assay.dtype)

float64


### Explanation

to_numeric returns a numerical Series when conversion is possible. errors="coerce" converts invalid entries to NaN. A float dtype is common because decimal percentages are present.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-12"></a>

## 12. Program 4: Correcting Text, Numeric, and Date Fields

### Problem statement

Prepare the working DataFrame so that text labels are consistent and numerical and date fields can be used correctly in comparisons, sorting, filtering, and summaries.

### Objective

To standardize text, correct important data types, and prepare a consistent clean_df for the remaining examples in the chapter.

### Pharmacy context

Type conversion changes how Python can process a value; it does not change the scientific meaning of the observation. For example, converting assay_percent to float allows calculations such as a mean or range, but it does not determine whether the assay result is acceptable. Similarly, a median-filled value is a documented classroom transformation and must not be presented as an original laboratory measurement.

### Algorithm

1. Continue with clean_df created after duplicate removal.
2. Remove extra spaces and standardize capitalization in product_name and dosage_form.
3. Remove the percent sign from assay_percent and convert measurement columns to numeric values using pd.to_numeric().
4. Convert manufacture_date and expiry_date to datetime values using mixed-format parsing and a documented day-first setting.
5. Fill selected numerical missing values with their column medians for this classroom dataset.
6. Replace a missing review_status with "Not Recorded".
7. Display the resulting data types to confirm that conversion was successful.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| str.strip() | Removes surrounding spaces | " ibuprofen " -> "ibuprofen" |
| str.title() | Standardizes simple label capitalization | "tablet" -> "Tablet" |
| errors="coerce" | Invalid values become missing | "not recorded" -> NaN |
| dayfirst | Controls day/month interpretation for ambiguous dates | True for 18-04-2026 |

### Important notes

Date parsing assumes day-first notation for ambiguous dates. The original raw file remains unchanged. Missing-value replacement is illustrative and changes descriptive statistics; real records require an approved, documented policy.

### Code and executed output

The output appears directly below the code cell.

In [13]:
# Continue with clean_df created in Program 3
clean_df["product_name"] = clean_df["product_name"].astype("string").str.strip().str.title()
clean_df["dosage_form"] = clean_df["dosage_form"].astype("string").str.strip().str.title()

clean_df["assay_percent"] = pd.to_numeric(
    clean_df["assay_percent"].astype("string").str.replace("%", "", regex=False),
    errors="coerce"
)

for col in ["dissolution_percent", "temperature_c", "humidity_percent"]:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

for col in ["manufacture_date", "expiry_date"]:
    clean_df[col] = pd.to_datetime(
        clean_df[col], errors="coerce", format="mixed", dayfirst=True
    )

for col in ["assay_percent", "dissolution_percent", "humidity_percent"]:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

clean_df["review_status"] = clean_df["review_status"].fillna("Not Recorded")

print(clean_df[[
    "assay_percent", "dissolution_percent",
    "manufacture_date", "expiry_date"
]].dtypes)

assay_percent                 Float64
dissolution_percent           float64
manufacture_date       datetime64[ns]
expiry_date            datetime64[ns]
dtype: object


### Explanation

The .str accessor applies text methods to a pandas string column. pd.to_numeric() and pd.to_datetime() convert text into data types that support numerical and date operations. errors="coerce" changes invalid entries to missing values, so the converted columns are checked and the documented classroom fill rules are applied only after conversion. The cleaned working DataFrame remains stored as clean_df for the following examples.

### Interpretation

Type conversion changes how Python can process a value; it does not change the scientific meaning of the observation. For example, converting assay_percent to float allows calculations such as a mean or range, but it does not determine whether the assay result is acceptable. Similarly, a median-filled value is a documented classroom transformation and must not be presented as an original laboratory measurement.

<a id="example-11-13"></a>

## 13. 11.10 Selecting Rows and Columns

### Problem statement

Demonstrate selecting Rows and Columns using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support selecting rows and columns and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `clean_df[['batch_id', 'product_name', 'assay_percent']].head(3)` in `selected`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [14]:
selected = clean_df[[
    "batch_id",
    "product_name",
    "assay_percent"
]].head(3)

print(selected.to_string(index=False))

batch_id product_name  assay_percent
 BPH-001  Paracetamol           99.2
 BPH-001  Paracetamol           98.7
 BPH-002    Ibuprofen           99.5


### Explanation

Double brackets return a DataFrame containing the listed columns. head(3) then displays the first three rows. Selection is useful for focusing an output on the variables relevant to the question.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-14"></a>

## 14. Program 5: Selecting a Compact Quality Review Table

### Problem statement

Create a compact DataFrame containing batch ID, product, assay, dissolution, site, and review status.

### Objective

To practise named-column selection without deleting information from the original DataFrame.

### Pharmacy context

A compact review table is useful when only a small set of variables is required for a specific classroom question. The value 99.5 shown for the formerly missing assay is the median-filled classroom value in clean_df. It should be documented as a transformed value rather than reported as an original measurement.

### Algorithm

1. Start from the cleaned DataFrame.
2. Create a Python list containing the required column names.
3. Use double-bracket selection to create review_df.
4. Display the first five records.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| review_columns | List of selected column names | 6 columns |
| review_df | Smaller DataFrame for reporting | Selected variables only |

### Code and executed output

The output appears directly below the code cell.

In [15]:
review_columns = [
    "batch_id",
    "product_name",
    "assay_percent",
    "dissolution_percent",
    "site",
    "review_status"
]

review_df = clean_df[review_columns]
print(review_df.head().to_string(index=False))

batch_id product_name  assay_percent  dissolution_percent  site review_status
 BPH-001  Paracetamol           99.2                 88.0 Delhi      Reviewed
 BPH-001  Paracetamol           98.7                 86.5 Delhi      Reviewed
 BPH-002    Ibuprofen           99.5                 82.0 Delhi       Pending
 BPH-002    Ibuprofen          100.4                 81.5 Delhi       Pending
 BPH-003   Cetirizine          101.1                 86.5 Noida      Reviewed


### Explanation

The list review_columns makes the selection easy to read and modify. clean_df[review_columns] returns the selected columns in the same order as the list. The original clean_df still contains all of its columns because this statement creates a separate review DataFrame.

### Interpretation

A compact review table is useful when only a small set of variables is required for a specific classroom question. The value 99.5 shown for the formerly missing assay is the median-filled classroom value in clean_df. It should be documented as a transformed value rather than reported as an original measurement.

<a id="example-11-15"></a>

## 15. 11.11 Conditional Filtering

### Problem statement

Demonstrate conditional Filtering using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support conditional filtering and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Create or calculate `delhi_tablets` from the values shown in the code.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [16]:
delhi_tablets = clean_df.loc[
    (clean_df["site"] == "Delhi")
    & (clean_df["dosage_form"] == "Tablet"),
    ["batch_id", "product_name", "site"]
]

print(len(delhi_tablets))

8


### Explanation

Both conditions must be True because & is used. loc returns only matching rows and selected columns. Parentheses keep each comparison together before pandas combines them.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-16"></a>

## 16. Program 6: Filtering Records for a Classroom Attention Screen

### Problem statement

Filter records where temperature is above 25 °C OR humidity is above 60%, and display selected review columns.

### Objective

To demonstrate Boolean OR filtering with two environmental parameters.

### Pharmacy context

The thresholds 25 °C and 60% are classroom screening values used only to demonstrate filtering. They do not define universal storage or stability limits. The key programming interpretation is that | implements an OR rule, while replacing it with & would require both conditions to be True and would therefore return fewer records.

### Algorithm

1. Start from the cleaned DataFrame.
2. Build one condition for temperature_c > 25.
3. Build a second condition for humidity_percent > 60.
4. Combine the conditions with | to represent OR.
5. Select identifiers, site, temperature, humidity, and status using loc.
6. Display the filtered records.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| temperature_c | Simulated recorded temperature | °C |
| humidity_percent | Simulated relative humidity | % |
| \| | Element-wise OR operator | Either condition may be True |
| loc | Select matching rows and named columns | Filtering method |

### Code and executed output

The output appears directly below the code cell.

In [17]:
attention = clean_df.loc[
    (clean_df["temperature_c"] > 25)
    | (clean_df["humidity_percent"] > 60),
    [
        "record_id",
        "product_name",
        "temperature_c",
        "humidity_percent",
        "site"
    ]
]

print(attention.to_string(index=False))

record_id product_name  temperature_c  humidity_percent     site
     R002  Paracetamol           25.5              57.0    Delhi
     R007  Amoxicillin           27.2              66.0 Gurugram
     R008  Amoxicillin           26.8              52.5 Gurugram
     R011 Azithromycin           28.0              70.0    Noida
     R012 Azithromycin           28.4              71.0    Noida
     R016   Amlodipine           25.1              61.0 Gurugram


### Explanation

The | operator keeps a row when either comparison is True. R002 appears because temperature is above the classroom boundary even though humidity is not. R016 appears because both comparisons are above the illustrative boundaries.

### Interpretation

The thresholds 25 °C and 60% are classroom screening values used only to demonstrate filtering. They do not define universal storage or stability limits. The key programming interpretation is that | implements an OR rule, while replacing it with & would require both conditions to be True and would therefore return fewer records.

<a id="example-11-17"></a>

## 17. 11.12 Sorting Data

### Problem statement

Demonstrate sorting Data using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support sorting data and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `clean_df.sort_values(by='assay_percent', ascending=True)` in `lowest_first`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [18]:
lowest_first = clean_df.sort_values(
    by="assay_percent",
    ascending=True
)

print(
    lowest_first[[
        "record_id",
        "product_name",
        "assay_percent"
    ]].head(3).to_string(index=False)
)

record_id product_name  assay_percent
     R016   Amlodipine           94.7
     R011 Azithromycin           96.8
     R012 Azithromycin           97.2


### Explanation

by identifies the sort key. ascending=True places smaller numerical values first. head(3) then shows the first three records after sorting.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-18"></a>

## 18. Program 7: Sorting by Site and Assay

### Problem statement

Arrange the cleaned records alphabetically by site and, within each site, from highest to lowest assay value.

### Objective

To demonstrate different ascending directions for two sort keys.

### Pharmacy context

Sorting is a presentation and workflow operation. A low value appearing first or last does not itself mean the observation is invalid. The parameters by and ascending should be chosen to make the intended review logic explicit.

### Algorithm

1. Start from the cleaned DataFrame.
2. Call sort_values with by=["site", "assay_percent"].
3. Set ascending=[True, False].
4. Select the first eight rows of the sorted result for display.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| by | Columns that determine row order | ["site", "assay_percent"] |
| ascending | Direction for each sort key | [True, False] |
| True | Ascending order | A-Z for site |
| False | Descending order | High-to-low assay |

### Code and executed output

The output appears directly below the code cell.

In [19]:
sorted_df = clean_df.sort_values(
    by=["site", "assay_percent"],
    ascending=[True, False]
)

print(
    sorted_df[[
        "site",
        "record_id",
        "product_name",
        "assay_percent"
    ]].head(8).to_string(index=False)
)

 site record_id product_name  assay_percent
Delhi      R009    Metformin          102.0
Delhi      R010    Metformin          101.5
Delhi      R019     Losartan          100.6
Delhi      R004    Ibuprofen          100.4
Delhi      R020     Losartan          100.2
Delhi      R013   Omeprazole          100.0
Delhi      R014   Omeprazole           99.6
Delhi      R003    Ibuprofen           99.5


### Explanation

The list in ascending has one Boolean for each column listed in by. Site is sorted A-Z, while assay is intended to be sorted high-to-low inside each site. The exact visible sequence depends on all matching rows and the stable handling of equal values.

### Interpretation

Sorting is a presentation and workflow operation. A low value appearing first or last does not itself mean the observation is invalid. The parameters by and ascending should be chosen to make the intended review logic explicit.

<a id="example-11-19"></a>

## 19. 11.13 Grouping and Aggregation

### Problem statement

Demonstrate grouping and Aggregation using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support grouping and aggregation and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `clean_df.groupby('site', as_index=False)['assay_percent'].mean()` in `site_mean`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [20]:
site_mean = clean_df.groupby(
    "site",
    as_index=False
)["assay_percent"].mean()

print(site_mean.round(2).to_string(index=False))

    site  assay_percent
   Delhi         100.17
Gurugram          99.05
   Noida          98.72


### Explanation

groupby("site") separates rows into site groups. Selecting assay_percent focuses the calculation on that measurement. mean() calculates one average per group. as_index=False keeps site as an ordinary column in the returned DataFrame.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-20"></a>

## 20. Program 8: Product-wise Aggregation of Assay Values

### Problem statement

Calculate count, mean, minimum, and maximum assay_percent for each product_name.

### Objective

To demonstrate groupby() with agg() and named output columns.

### Pharmacy context

The summary describes only the simulated dataset. For example, Amlodipine has a mean of 99.95% across two classroom records, but the range from 94.7% to 105.2% shows why a mean alone can hide spread. Parameter choices determine what is summarized: changing the group column, aggregation function, or cleaning rule changes the result and must therefore be reported.

### Algorithm

1. Start from the cleaned DataFrame.
2. Group records by product_name.
3. Use agg with named outputs for record_count, mean_assay, min_assay, and max_assay.
4. Reset or avoid a hierarchical index using as_index=False.
5. Round numerical summaries to two decimal places.
6. Display the summary.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| product_name | Grouping variable | Medicine name |
| record_count | Number of assay observations | count |
| mean_assay | Average assay in group | % |
| min_assay | Smallest assay in group | % |
| max_assay | Largest assay in group | % |

### Code and executed output

The output appears directly below the code cell.

In [21]:
product_summary = clean_df.groupby(
    "product_name",
    as_index=False
).agg(
    record_count=("assay_percent", "count"),
    mean_assay=("assay_percent", "mean"),
    min_assay=("assay_percent", "min"),
    max_assay=("assay_percent", "max")
)

summary_cols = [
    "mean_assay",
    "min_assay",
    "max_assay"
]
product_summary[summary_cols] = (
    product_summary[summary_cols].round(2)
)

print(product_summary.head().to_string(index=False))

product_name  record_count  mean_assay  min_assay  max_assay
  Amlodipine             2       99.95       94.7      105.2
 Amoxicillin             2       98.15       97.9       98.4
Azithromycin             2        97.0       96.8       97.2
  Cetirizine             1       101.1      101.1      101.1
   Ibuprofen             2       99.95       99.5      100.4


### Explanation

Named aggregation syntax gives each output column a clear name. count operates on non-missing assay values. Because missing assay has already been filled in the demonstration clean dataset, the count equals the number of retained records for those products.

### Interpretation

The summary describes only the simulated dataset. For example, Amlodipine has a mean of 99.95% across two classroom records, but the range from 94.7% to 105.2% shows why a mean alone can hide spread. Parameter choices determine what is summarized: changing the group column, aggregation function, or cleaning rule changes the result and must therefore be reported.

<a id="example-11-21"></a>

## 21. 11.13.1 Multiple Grouping Columns

### Problem statement

Demonstrate multiple Grouping Columns using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support multiple grouping columns and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Store `clean_df.groupby(['site', 'dosage_form'], as_index=False).size()` in `site_form_count`.
2. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [22]:
site_form_count = clean_df.groupby(
    ["site", "dosage_form"],
    as_index=False
).size()

print(site_form_count.head().to_string(index=False))

    site dosage_form  size
   Delhi     Capsule     2
   Delhi      Tablet     8
Gurugram     Capsule     2
Gurugram      Tablet     2
   Noida      Tablet     5


### Explanation

A list of grouping columns creates groups defined by the combination of both variables. size() counts rows in each observed group. Only combinations actually present in the data are normally returned.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-22"></a>

## 22. 11.14 Exporting Cleaned Datasets

### Problem statement

Demonstrate exporting Cleaned Datasets using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support exporting cleaned datasets and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Import `pathlib`.
2. Execute `OUTPUT_DIR.mkdir(exist_ok=True)`.
3. Execute `clean_df.to_csv(OUTPUT_DIR / 'pharma_quality_clean.csv', index=False)`.
4. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [23]:
from pathlib import Path

OUTPUT_DIR.mkdir(exist_ok=True)

clean_df.to_csv(
    OUTPUT_DIR / "pharma_quality_clean.csv",
    index=False
)
print("File created:", (OUTPUT_DIR / "pharma_quality_clean.csv").is_file())

File created: True


### Explanation

mkdir(exist_ok=True) creates the output folder if it is missing. to_csv writes the DataFrame. index=False omits the row-number index so that only intended dataset columns are exported.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

<a id="example-11-23"></a>

## 23. Program 9: Exporting a Cleaned Pharmaceutical Dataset

### Problem statement

Create an output folder, export the cleaned dataset to CSV, and confirm that the file exists.

### Objective

To demonstrate a reproducible export step that protects the original source file.

### Pharmacy context

Raw and cleaned datasets should be kept as separate files so that transformations can be audited and repeated. A filename such as pharma_quality_clean.csv communicates that the file is processed; in larger projects, a date, version, or data-dictionary reference may also be appropriate.

### Algorithm

1. Import Path from pathlib.
2. Create the output folder if required.
3. Define a descriptive output file path.
4. Export clean using to_csv with index=False.
5. Check the file with Path.exists().

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| output_dir | Folder for cleaned results | output |
| output_file | Path of exported CSV | output/pharma_quality_clean.csv |
| index=False | Do not write DataFrame index | CSV parameter |
| exists() | Checks whether file is present | True / False |

### Code and executed output

The output appears directly below the code cell.

In [24]:
from pathlib import Path

output_dir = OUTPUT_DIR
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "pharma_quality_clean.csv"

clean_df.to_csv(
    output_file,
    index=False
)

print(output_file)
print("File created:", output_file.exists())

/workspace/scratch/1ff152582af4/notebook_work/execution/chapter_11/outputs/chapter_11/pharma_quality_clean.csv
File created: True


### Explanation

Path objects make file construction readable and portable. mkdir(exist_ok=True) avoids an error when the folder already exists. index=False prevents an unnecessary index column from appearing in the CSV.

### Interpretation

Raw and cleaned datasets should be kept as separate files so that transformations can be audited and repeated. A filename such as pharma_quality_clean.csv communicates that the file is processed; in larger projects, a date, version, or data-dictionary reference may also be appropriate.

<a id="example-11-24"></a>

## 24. Program 10: Complete Cleaning, Validation, Summary, and Export

### Problem statement

Produce a cleaned DataFrame, validate the final row and missing-value counts, calculate a site-level assay summary, and save the cleaned data.

### Objective

To combine import, standardization, type conversion, missing-value handling, duplicate removal, grouping, and export in one reproducible notebook workflow.

### Pharmacy context

The final shape of 19 x 13 confirms removal of one repeated record_id. The site-level means depend on the earlier cleaning choices, especially median filling and duplicate removal. Therefore, summaries must be interpreted together with the documented transformation rules. Changing errors, subset, keep, or the fill statistic can change the numerical output.

### Algorithm

1. Read the raw CSV file.
2. Standardize product and dosage-form text.
3. Convert assay, dissolution, humidity, and temperature to numeric values.
4. Convert date columns to datetime.
5. Remove later occurrences of duplicate record_id values.
6. Fill selected numeric missing values with their column medians.
7. Replace missing review status with "Not Recorded".
8. Validate row count and remaining missing values.
9. Group by site and calculate count and mean assay.
10. Export the cleaned DataFrame to output/pharma_quality_clean.csv.

### Variables and parameters

| Variable / Parameter | Meaning | Example / Unit |
| --- | --- | --- |
| raw_df | Imported raw data | 20 x 13 |
| final_df | Cleaned DataFrame | 19 x 13 |
| errors="coerce" | Invalid conversion becomes missing | Conversion rule |
| subset=["record_id"] | Defines duplicate key | Duplicate rule |
| median() | Classroom fill statistic | Per selected numeric column |
| index=False | Prevents extra index column | Export parameter |

### Important notes

Date parsing assumes day-first notation for ambiguous dates. The original raw file remains unchanged. Missing-value replacement is illustrative and changes descriptive statistics; real records require an approved, documented policy.

### Code and executed output

The output appears directly below the code cell.

In [25]:
from pathlib import Path
import pandas as pd

final_df = pd.read_csv(DATA_DIR / "pharma_quality_raw.csv")
final_df["product_name"] = final_df["product_name"].astype("string").str.strip().str.title()
final_df["dosage_form"] = final_df["dosage_form"].astype("string").str.strip().str.title()
final_df["assay_percent"] = pd.to_numeric(final_df["assay_percent"].astype("string").str.replace("%", "", regex=False), errors="coerce")
for col in ["dissolution_percent", "temperature_c", "humidity_percent"]:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")
final_df["manufacture_date"] = pd.to_datetime(final_df["manufacture_date"], errors="coerce", format="mixed", dayfirst=True)
final_df["expiry_date"] = pd.to_datetime(final_df["expiry_date"], errors="coerce", format="mixed", dayfirst=True)
final_df = final_df.drop_duplicates(subset=["record_id"], keep="first").reset_index(drop=True)
for col in ["assay_percent", "dissolution_percent", "humidity_percent"]:
    final_df[col] = final_df[col].fillna(final_df[col].median())
final_df["review_status"] = final_df["review_status"].fillna("Not Recorded")

site_summary = final_df.groupby("site", as_index=False).agg(records=("record_id", "count"), mean_assay=("assay_percent", "mean"))
site_summary["mean_assay"] = site_summary["mean_assay"].round(2)
OUTPUT_DIR.mkdir(exist_ok=True)
final_df.to_csv(OUTPUT_DIR / "pharma_quality_clean.csv", index=False)
print("Final shape:", final_df.shape)
print(site_summary.to_string(index=False))

Final shape: (19, 13)
    site  records  mean_assay
   Delhi       10      100.17
Gurugram        4       99.05
   Noida        5       98.72


### Explanation

The program follows a deliberate order: standardize and convert first, remove the known duplicate key, fill selected missing numerical values, validate, summarize, and export. A separate output file is created. Both date columns use format="mixed" to accept the deliberately varied classroom date strings while dayfirst=True correctly interprets entries such as 18-04-2026.

### Interpretation

The final shape of 19 x 13 confirms removal of one repeated record_id. The site-level means depend on the earlier cleaning choices, especially median filling and duplicate removal. Therefore, summaries must be interpreted together with the documented transformation rules. Changing errors, subset, keep, or the fill statistic can change the numerical output.

<a id="example-11-25"></a>

## 25. 11.16 Validation After Cleaning

### Problem statement

Demonstrate validation After Cleaning using the simulated values or objects in the cell below.

### Objective

Understand how the Python operations in this example support validation after cleaning and interpret the result correctly.

### Pharmacy context

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

### Algorithm

1. Check `final_df['record_id'].is_unique` before reporting success.
2. Check `final_df['assay_percent'].notna().all()` before reporting success.
3. Display the results for inspection.

### Code and executed output

The output appears directly below the code cell.

In [26]:
assert final_df["record_id"].is_unique, (
    "record_id is not unique"
)
assert final_df["assay_percent"].notna().all(), (
    "assay still contains missing values"
)
print("Validation checks passed")

Validation checks passed


### Explanation

assert stops execution if a required condition is False. is_unique checks whether every identifier occurs once. notna().all() checks that the selected column has no missing values. Assertions are useful for classroom reproducibility but should be designed to reflect justified rules.

### Interpretation

Cleaning changes the analytical representation of data. Keep raw data, document duplicates and conversion failures, and flag imputation. Median replacement here is a teaching choice, not a rule for real QC or clinical records.

## Review and practice

- Change one simulated input and predict the effect before rerunning the relevant cell.
- State the unit and meaning of each reported result.
- Explain one input condition or limitation that matters for the example.
- Restart the kernel and run all cells to check that the notebook is reproducible.

## Source and validation

Adapted from **Chapter_11_Data_Cleaning_and_Basic_Analysis_BPharm_Revised.docx**. Worked programs and the complete code demonstrations are retained in chapter order. Installation commands are optional instructions; intentionally broken debugging examples are shown as corrected solutions. Notebook-specific changes are recorded in `Notebook_Guide_and_Validation.md`.